In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 98 — Memory Benchmark Project

## Goal
Compare three memory strategies for conversational agents:
**short-term** (buffer), **long-term** (summary), and
**retrieval** (vector search) — measuring each on accuracy,
context utilization, and relevance.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Buffer memory** | Keep last N messages verbatim |
| **Summary memory** | LLM summarizes conversation history |
| **Retrieval memory** | Store + retrieve via vector similarity |
| **Evaluation metrics** | Compare strategies on same queries |

## Stack
- `langchain-ollama` — LLM
- `chromadb` — vector store for retrieval memory
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, json, time
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
import chromadb

llm = ChatOllama(model="qwen3.5:9b", temperature=0)
print("All imports OK")

## Step 1 — Simulated Conversation History

We create a long conversation with specific facts the agent
should remember. Then we test recall with targeted questions.

In [ ]:
# A realistic multi-turn conversation with embedded facts
CONVERSATION = [
    {"role": "user", "content": "Hi, I'm planning a trip to Japan for 2 weeks in April."},
    {"role": "assistant", "content": "Great choice! April is cherry blossom season in Japan."},
    {"role": "user", "content": "My budget is $5,000 total. I'll be traveling with my wife Sarah."},
    {"role": "assistant", "content": "$5,000 for two people for 2 weeks is doable if you mix hotels and hostels."},
    {"role": "user", "content": "We want to visit Tokyo, Kyoto, and Osaka. Sarah is vegetarian."},
    {"role": "assistant", "content": "Japan has great vegetarian options, especially temple cuisine (shojin ryori)."},
    {"role": "user", "content": "I'm allergic to shellfish, so we need to be careful with that."},
    {"role": "assistant", "content": "Good to note. Many Japanese broths use bonito (fish), but shellfish is easier to avoid."},
    {"role": "user", "content": "We fly from New York JFK on April 3rd."},
    {"role": "assistant", "content": "That should put you in Tokyo around April 4th with the time difference."},
    {"role": "user", "content": "Our hotel in Tokyo is the Shinjuku Granbell. Booked for 5 nights."},
    {"role": "assistant", "content": "Nice! The Granbell is well-located near Shinjuku station."},
    {"role": "user", "content": "Then we take the Shinkansen to Kyoto on April 9th."},
    {"role": "assistant", "content": "The bullet train takes about 2 hours 15 minutes. Get a JR Pass!"},
    {"role": "user", "content": "We're staying at a traditional ryokan in Kyoto called Gion Hatanaka."},
    {"role": "assistant", "content": "Lovely! Gion is the geisha district. The ryokan experience is unforgettable."},
    {"role": "user", "content": "Sarah really wants to see the Fushimi Inari shrine."},
    {"role": "assistant", "content": "Go early morning (6-7am) to avoid crowds at the torii gates."},
    {"role": "user", "content": "What about day trips from Kyoto?"},
    {"role": "assistant", "content": "Nara (deer park) and Osaka (street food) are easy day trips."},
]

# Test questions that require recalling specific facts
TEST_QUESTIONS = [
    {"question": "What is my travel companion's name?", "answer": "Sarah"},
    {"question": "What is my total budget?", "answer": "$5,000"},
    {"question": "What food allergy do I have?", "answer": "shellfish"},
    {"question": "What hotel are we staying at in Tokyo?", "answer": "Shinjuku Granbell"},
    {"question": "When do we take the Shinkansen to Kyoto?", "answer": "April 9th"},
    {"question": "What's the name of our ryokan in Kyoto?", "answer": "Gion Hatanaka"},
    {"question": "Which shrine does Sarah especially want to visit?", "answer": "Fushimi Inari"},
    {"question": "Which airport do we depart from?", "answer": "JFK"},
]

print(f"Conversation: {len(CONVERSATION)} messages")
print(f"Test questions: {len(TEST_QUESTIONS)}")

## Step 2 — Strategy 1: Short-Term (Buffer) Memory

Keep only the last `N` messages. Simple but loses early context.

In [ ]:
def buffer_memory_answer(conversation, question, buffer_size=6):
    """Answer using only the last N messages as context."""
    recent = conversation[-buffer_size:]
    
    messages = [SystemMessage(content="Answer based ONLY on the conversation history. "
                              "If the answer isn't in the history, say 'NOT FOUND'. "
                              "Be brief. No thinking tags.")]
    for msg in recent:
        if msg["role"] == "user":
            messages.append(HumanMessage(content=msg["content"]))
        else:
            messages.append(AIMessage(content=msg["content"]))
    messages.append(HumanMessage(content=question))
    
    resp = llm.invoke(messages)
    text = resp.content
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text

print("Strategy 1: Buffer memory (last 6 messages)")

## Step 3 — Strategy 2: Summary Memory

LLM creates a running summary of the entire conversation.

In [ ]:
def create_summary(conversation):
    """Create a summary of the full conversation."""
    conv_text = "\n".join(f"{m['role']}: {m['content']}" for m in conversation)
    
    msg = llm.invoke([
        SystemMessage(content="""Summarize this conversation, preserving ALL specific facts:
names, dates, numbers, places, hotels, dietary requirements, allergies, etc.
Be comprehensive. 200-300 words. No thinking tags."""),
        HumanMessage(content=conv_text)
    ])
    text = msg.content
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text

def summary_memory_answer(summary_text, question):
    """Answer using the conversation summary."""
    messages = [
        SystemMessage(content="Answer based ONLY on this conversation summary. "
                      "If the answer isn't there, say 'NOT FOUND'. "
                      "Be brief. No thinking tags."),
        HumanMessage(content=f"Summary:\n{summary_text}\n\nQuestion: {question}")
    ]
    resp = llm.invoke(messages)
    text = resp.content
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text

print("Creating conversation summary...")
summary = create_summary(CONVERSATION)
print(f"Summary ({len(summary)} chars):\n{summary[:300]}...")

## Step 4 — Strategy 3: Retrieval Memory (Vector Search)

Store each message in ChromaDB, retrieve relevant ones per query.

In [ ]:
# Build retrieval memory
chroma_client = chromadb.Client()

# Delete collection if it exists (for re-runs)
try:
    chroma_client.delete_collection("memory_98")
except Exception:
    pass

collection = chroma_client.create_collection("memory_98")

# Store each exchange as a document
for i in range(0, len(CONVERSATION) - 1, 2):
    user_msg = CONVERSATION[i]["content"]
    asst_msg = CONVERSATION[i + 1]["content"]
    combined = f"User: {user_msg}\nAssistant: {asst_msg}"
    collection.add(
        documents=[combined],
        ids=[f"turn_{i//2}"],
        metadatas=[{"turn": i // 2}]
    )

print(f"Stored {collection.count()} conversation turns in ChromaDB")

def retrieval_memory_answer(question, n_results=3):
    """Answer using retrieved relevant conversation turns."""
    results = collection.query(query_texts=[question], n_results=n_results)
    context = "\n---\n".join(results["documents"][0])
    
    messages = [
        SystemMessage(content="Answer based ONLY on these retrieved conversation excerpts. "
                      "If the answer isn't there, say 'NOT FOUND'. "
                      "Be brief. No thinking tags."),
        HumanMessage(content=f"Retrieved context:\n{context}\n\nQuestion: {question}")
    ]
    resp = llm.invoke(messages)
    text = resp.content
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text

print("Strategy 3: Retrieval memory ready")

## Step 5 — Run the Benchmark

In [ ]:
def check_answer(response, expected):
    """Check if the expected answer appears in the response."""
    return expected.lower() in response.lower()

results = {"buffer": [], "summary": [], "retrieval": []}

print("Running benchmark across all strategies...\n")
for i, tq in enumerate(TEST_QUESTIONS):
    q = tq["question"]
    expected = tq["answer"]
    print(f"Q{i+1}: {q} (expected: {expected})")
    
    # Buffer memory
    buf_ans = buffer_memory_answer(CONVERSATION, q)
    buf_correct = check_answer(buf_ans, expected)
    results["buffer"].append({"q": q, "answer": buf_ans, "correct": buf_correct})
    
    # Summary memory
    sum_ans = summary_memory_answer(summary, q)
    sum_correct = check_answer(sum_ans, expected)
    results["summary"].append({"q": q, "answer": sum_ans, "correct": sum_correct})
    
    # Retrieval memory
    ret_ans = retrieval_memory_answer(q)
    ret_correct = check_answer(ret_ans, expected)
    results["retrieval"].append({"q": q, "answer": ret_ans, "correct": ret_correct})
    
    b = "✅" if buf_correct else "❌"
    s = "✅" if sum_correct else "❌"
    r = "✅" if ret_correct else "❌"
    print(f"   Buffer: {b}  Summary: {s}  Retrieval: {r}")

print("\nBenchmark complete!")

In [ ]:
# Score summary
print("=" * 60)
print("MEMORY BENCHMARK RESULTS")
print("=" * 60)

for strategy in ["buffer", "summary", "retrieval"]:
    correct = sum(1 for r in results[strategy] if r["correct"])
    total = len(results[strategy])
    pct = 100 * correct / total
    bar = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    print(f"\n  {strategy.upper():12s}: {correct}/{total} ({pct:.0f}%) {bar}")
    
    # Show which questions each strategy got wrong
    wrong = [r for r in results[strategy] if not r["correct"]]
    for w in wrong:
        print(f"    ❌ {w['q'][:50]} → {w['answer'][:50]}")

print("\n" + "=" * 60)
print("ANALYSIS")
print("=" * 60)
print("Buffer memory: Fast but loses early context (old facts forgotten)")
print("Summary memory: Good recall if summary captures details")
print("Retrieval memory: Best for specific fact lookup, may miss context")

## 🧠 Key Concepts Recap

| Strategy | Pros | Cons |
|----------|------|------|
| **Buffer** | Simple, fast, low cost | Loses old context |
| **Summary** | Compact, captures themes | May lose specific details |
| **Retrieval** | Precise fact recall | Depends on embedding quality |

## When to Use Each
- **Buffer**: Simple chatbots, short conversations
- **Summary**: Long conversations where themes matter
- **Retrieval**: Fact-heavy conversations (e.g., travel planning)
- **Hybrid**: Combine summary + retrieval for best of both

## Extending This Project
- Hybrid strategy (summary for context + retrieval for facts)
- Vary buffer size and measure impact
- Test with longer conversations (100+ turns)
- Compare embedding models for retrieval quality
- Add latency and cost metrics